# W05 · 01 조절기 3종 — 같은 LJ 유체 · 같은 목표 온도 · 다른 요동
목표 3
1. W03 LJ 유체 64입자(ε 0.109 · σ 3.40 · Δt 1 fs)에 velocity rescale · Berendsen(τ 100 fs) · Langevin(γ 0.01/fs) 을 붙여 각 2,000스텝 NVT
2. 후반 1,000스텝의 σ_T 를 정규 앙상블 기대값 T₀√(2/3N) = 30.6 K 와 비교 (슬라이드 10 · 12)
3. 결과 표 → `out/thermostat_sigma.csv`

In [ ]:
# 기본 = 로컬 mdai · Claude Code 실행(아래 if 는 건너뜀) · Colab 은 선택: 저장소 clone 후 week05 로 이동
import os, sys, json, math
if "google.colab" in sys.modules and not os.path.exists("scripts/md_stats.py"):
    !git clone https://github.com/sentirain7/2026_MOOC_Lecture_Materials.git repo && %cd repo/week05
import numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "scripts")
from md_stats import *   # LJFluid thermostats equilibration block_se autocorr OUT

In [ ]:
rows = thermostats(T0=300.0, steps=2000)     # 약 1~2분
for r in rows: print(f'{r["thermostat"]:10s} 평균 {r["T_mean_tail"]:7.2f} K  σ_T {r["sigma_T_tail"]:6.2f} K  기대 {r["sigma_T_expected_canonical"]:5.2f} K  도달 {r["reach_step"]}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for name in ["none", "rescale", "berendsen", "langevin"]:
    T = np.loadtxt(OUT / f"T_{name}.csv", skiprows=1); ax.plot(T, lw=1, label=name)
ax.axhline(300, color="gray", ls="--", lw=0.8); ax.set_xlabel("step (dt 1 fs)"); ax.set_ylabel("T (K)"); ax.legend(ncol=4)
fig.tight_layout(); fig.savefig(OUT / "thermostats.png", dpi=120); plt.show()

## 읽기
- rescale: σ_T = 0 — 매 스텝 T 를 T₀ 로 고정 · 정규 앙상블이 아니다 · 초기 가열 · 냉각 전용
- Berendsen: 빨리 붙지만 σ_T ≈ 9 K ≪ 30.6 K — 열욕이 요동까지 흡수(열용량 편향) · 평형화 전용
- Langevin: σ_T ≈ 30 K ≈ 기대값 — 정규 앙상블 재현 · 단 마찰 γ 가 확산 · 점도를 왜곡
- NVE(none): 조절기 없음 · 평균이 T₀ 와 다르다(격자 → 등분배) · E 보존 확인용

Nosé–Hoover 는 LAMMPS `fix nvt` 로 W06 에서 직접 돌린다(여기서는 구현하지 않음).